<a href="https://colab.research.google.com/github/comp0161/tutorials/blob/main/comp0161_2026_lab2_synthesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP0161 Auditory Computing Week 2: Digital Audio Synthesis

In this tutorial we will work through some simple audio synthesis. The techniques are rudimentary; much richer and more capable systems exist, but they are also more complex and less transparent. You may never need to do anything like this again, but doing it once will hopefully be a learning experience.

# Setting Up

We'll mostly be using standard mathematical packages that are installed by default on Colab. But later on we're going to want to use some without having to implement everything from scratch, so we'll install Spotify's [pedalboard](https://github.com/spotify/pedalboard) library for that functionality.

In [ ]:
%pip install --quiet pedalboard

Import libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.display import Image, Audio

# we'll be wanting some random numbers later
# we create a generator with a fixed seed for consistent behaviour
# if you want different results, change the seed
SEED = 9907
rng = np.random.default_rng(SEED)

import pedalboard

## Configuration

We define some global variables that will be used as the default arguments to our functions. In general these can be overridden at call time, but we typically want them to be consistent throughout a session so it's convenient to centralise them here and just let them default. Feel free to experiment with different values.

In [ ]:
SAMPLE_RATE = 44100
PITCH = 440
PITCHES = [55, 110, 220, 440, 880]
DURATION = 1
RAND_PHASE = 0

# Playing Audio

We're going to use simple NumPy arrays of numbers to represent sounds in Pulse Code Modulation form. The `IPython.display.Audio` class imported above provides the ability to embed such data in the page in playable form. Here we just provide a simple wrapper function around that.

In [ ]:
def play(x, rate=SAMPLE_RATE, dupe_stereo=True):
    """
    Display a numpy array using IPython.display.Audio.
    Optionally duplicates mono to stereo (on by default).
    """
    if dupe_stereo and (len(x.shape) == 1):
        x = np.stack((x,x)).copy()

    display(Audio(x, rate=rate))

# Oscillators

The basic building block for all sounds is a simple sinusoidal oscillation, which is easily generated using NumPy's `sin` function. For reasons we'll get to in a bit, we provide the ability to randomise the phase as well as just specifying it explicitly.

Note that the one thing we don't provide an argument for here is amplitude. The waves produced here will always oscillate in the range [-1, 1]. We'll deal with scaling later.

In [ ]:
def tone(hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, phase=0, random_phase=RAND_PHASE):
    """
    Generate a simple pure sine tone of specified frequency
    and duration.
    """
    return np.sin(rng.uniform() * random_phase * 2 * np.pi + phase + hz * 2 * np.pi * np.arange(int(duration * rate))/rate)

Let's visualise and listen to this waveform. (Note that for visualisation we choose a much shorter duration so that we can actually see the waveform.)

In [ ]:
plt.plot(tone(duration=3/PITCH))
play(tone())

A single sine tone isn't all that interesting, but we can make more complex sound by taking a weighted sum of such sines.

In [ ]:
def tones(hz=PITCHES, duration=DURATION, weights=None, rate=SAMPLE_RATE, phases=None, random_phase=RAND_PHASE):
    """
    Generate a (potentially weighted) sum of pure sine tones.
    """
    if weights is None:
        weights = np.ones(len(hz))

    if phases is None:
        phases = np.zeros(len(hz))

    result = weights[0] * tone(hz[0], duration, rate, phases[0], random_phase)
    for ii in range(1, len(hz)):
        result += weights[ii] * tone(hz[ii], duration, rate, phases[ii], random_phase)

    return result

Once again, let's look and listen.

In [ ]:
plt.plot(tones(duration=3/PITCH))
play(tones())

As discussed in Monday's lecture, there are some useful harmonically-rich waveforms that we can make out of sines. These are easy to generate in an idealised form, but the result contains infinite harmonics that lead to aliasing, introducing harsh artefacts into the sound.

In [ ]:
def hard_saw(hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, phase=0, random_phase=RAND_PHASE):
  """
  Generate a non-bandlimited saw wave with the same parameters
  as our basic tone. Note that the interpretation of random_phase
  here is different from the band_limited versions later, since
  we're not explicitly separating the harmonics.
  """
  ramp = rng.uniform() * random_phase + phase + 0.5 + hz * np.arange(int(duration * rate))/rate
  return np.modf(ramp)[0] * 2 - 1

def hard_square(hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, phase=0, random_phase=RAND_PHASE):
  """
  Generate a non-bandlimited square wave with the same parameters
  as our basic tone. (We'll actually use that function here, which
  is inefficient but easy.) Note that the interpretation of random_phase
  here is different from the band_limited versions later, since
  we're not explicitly separating the harmonics.
  """
  return (tone(hz, duration, rate, phase, random_phase) > 0).astype(float) * 2 - 1

def hard_triangle(hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, phase=0, random_phase=RAND_PHASE):
  """
  Generate a non-bandlimited triangle wave with the same parameters
  as our basic tone. Note that the interpretation of random_phase
  here is different from the band_limited versions later, since
  we're not explicitly separating the harmonics.
  """
  return 2 * np.abs(hard_saw(hz, duration, rate, phase + 0.25, random_phase)) - 1

These produce nice clean plots, but there is an audible roughness to the sound.

In [ ]:
plt.plot(hard_saw(duration=3/PITCH))
plt.title('Sawtooth Wave')
play(hard_saw())
plt.figure()
plt.plot(hard_square(duration=3/PITCH))
plt.title('Square Wave')
play(hard_square())
plt.figure()
plt.plot(hard_triangle(duration=3/PITCH))
plt.title('Triangle Wave')
play(hard_triangle(duration=3))

We can plot the spectra of these waves and observe the aliased higher harmonics.

In [ ]:
plt.magnitude_spectrum(hard_saw(), Fs=SAMPLE_RATE, scale='dB');
plt.title('Sawtooth Wave')
plt.figure()
plt.magnitude_spectrum(hard_square(), Fs=SAMPLE_RATE, scale='dB');
plt.title('Square Wave')
plt.figure()
plt.magnitude_spectrum(hard_triangle(), Fs=SAMPLE_RATE, scale='dB');
plt.title('Triangle Wave');

Instead, let's create **bandlimited** versions of these waves by means of **additive synthesis**.

We can use the recipes from the lecture:

$$
\text{saw} = -\sum_{k=1}^{N} (-1)^k \frac{f_k}{k}
$$

$$
\text{square} = \sum_{k=1}^{N} \frac{f_{2k-1}}{2k-1}
$$

$$
\text{triangle} = -\sum_{k=1}^{N} (-1)^k \frac{f_{2k-1}}{(2k-1)^2}
$$

In the ideal waveforms, these sums would go to infinity. But we want to impose a bandlimit, so we'll only include a finite number of components, $N$. We'll choose this to be within the Nyquist limit.

<details>
<summary>Note</summary>
These formulae often also include a scaling factor to keep the output in some desired range. In the functions below we instead normalise the wave after generating the sum.
</details>

In [ ]:
def saw (hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, max_harm=10000, random_phase=RAND_PHASE ):
    """
    Generate a band-limited sawtooth wave of specified frequency
    and duration. Phases of the individual harmonics can be randomised
    by an amount specified by `random_phase` (interpreted as
    largest fraction of a cycle).
    """
    assert(hz < rate/2)

    N = int(duration * SAMPLE_RATE)

    angles = 2 * np.pi * np.arange(N)/rate
    if max_harm is None: max_harm = 10000
    max_harm = np.min([int(np.floor(rate / (2 * hz))), max_harm])

    # start with the fundamental
    result = np.sin(angles * hz)
    components = 1

    # add the harmonics
    for harm in range(2, max_harm + 1):
        result -= ((-1)**(harm)) * np.sin(angles * hz * harm + rng.uniform() * random_phase * 2 * np.pi ) / harm
        components += 1

    # scale into [-1, 1]
    result = 2 * (result - np.min(result))/(np.max(result) - np.min(result)) - 1

    return result

def square (hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, max_harm=10000, random_phase=RAND_PHASE ):
    """
    Generate a band-limited square wave of specified frequency
    and duration. Phases of the individual harmonics can be randomised
    by an amount specified by `random_phase` (interpreted as
    largest fraction of a cycle).
    """
    assert(hz < rate/2)

    N = int(duration * SAMPLE_RATE)

    angles = 2 * np.pi * np.arange(N)/rate

    if max_harm is None: max_harm = 10000
    max_harm = np.min([int(np.floor(rate / (2 * hz))), max_harm])

    # start with the fundamental
    result = np.sin(angles * hz)

    components = 1

    # add the harmonics
    for harm in range(3, max_harm + 1, 2):
        result += np.sin(angles * hz * harm + rng.uniform() * random_phase * 2 * np.pi ) / harm
        components += 1

    # scale into [-1, 1]
    result = 2 * (result - np.min(result))/(np.max(result) - np.min(result)) - 1

    return result

def triangle (hz=PITCH, duration=DURATION, rate=SAMPLE_RATE, max_harm=10000, random_phase=RAND_PHASE ):
    """
    Generate a band-limited triangle wave of specified frequency
    and duration. Phases of the individual harmonics can be randomised
    by an amount specified by `random_phase` (interpreted as
    largest fraction of a cycle).
    """
    assert(hz < rate/2)

    N = int(duration * SAMPLE_RATE)

    angles = 2 * np.pi * np.arange(N)/rate

    if max_harm is None: max_harm = 10000
    max_harm = np.min([int(np.floor(rate / (2 * hz))), max_harm])

    # start with the fundamental
    result = np.sin(angles * hz)

    components = 1

    # add the harmonics
    for harm in range(3, max_harm + 1, 2):
        result -= ((-1)**(components+1)) * np.sin(angles * hz * harm + rng.uniform() * random_phase * 2 * np.pi ) / (harm * harm)
        components += 1

    # scale into [-1, 1]
    result = 2 * (result - np.min(result))/(np.max(result) - np.min(result)) - 1

    return result

Check 'em out:

In [ ]:
plt.plot(saw(duration=3/PITCH))
plt.title('Bandlimited Saw')
play(saw())
plt.figure()
plt.plot(square(duration=3/PITCH))
plt.title('Bandlimited Square')
play(square())
plt.figure()
plt.plot(triangle(duration=3/PITCH))
plt.title('Bandlimited Triangle')
play(triangle())

Now the spectra no longer contain aliased harmonics:

In [ ]:
plt.magnitude_spectrum(saw(), Fs=SAMPLE_RATE, scale='dB');
plt.title('Bandlimited Saw')
plt.figure()
plt.magnitude_spectrum(square(), Fs=SAMPLE_RATE, scale='dB');
plt.title('Bandlimited Square')
plt.figure()
plt.magnitude_spectrum(triangle(), Fs=SAMPLE_RATE, scale='dB');
plt.title('Bandlimited Triangle');

# Noise

Not all sounds are **harmonic** (ie, containing only integer multiples of a single fundamental frequency) -- indeed, in practice most sounds will have some inharmonic content just due to the vagaries of material reality. This is particularly true of **percussive** sounds produced by objects that have more degrees of vibrational freedom, like drum membranes and cymbals.

To emulate this sort of content, we can make use of **noise** sources, which contain mixtures of frequencies that are not harmonically related.

Noise is often characterised in terms of the "colour" of its spectrum, by analogy with the distribution of frequencies in light.

* **white** noise contains roughly equal amounts of every frequency
* **pink** noise contains more low frequencies, with the distribution going as $1/f^{\alpha}$ for some parameter $\alpha$, typically in the range $0 < \alpha \le 2$. Larger $\alpha$ shifts the spectrum more steeply towards lower frequencies, which are sometimes also referred to as **brown** or **Brownian** noise. Because this noise is dominated by low frequencies it produces something akin to a **random walk**.
* **blue** noise contains more high frequencies, with the distribution typically going as $f^{\alpha}$, again with $\alpha$ increasing the slope. Heavily shifted blue noise is sometimes referred to as **violet**.

We'll create all of these in the same way, starting with white noise and passing a helper function to do any desired shaping:

In [ ]:
def noise(duration=DURATION, rate=SAMPLE_RATE, shape=None):
    """
    Generate N noise samples with the power spectrum
    optionally shaped by the supplied function.
    """
    N = int(duration * SAMPLE_RATE)
    noise_spectrum = np.fft.rfft(rng.standard_normal(N))

    if shape is not None:
        shape_spectrum = shape(np.fft.rfftfreq(N))
        # normalise to preserve energy
        shape_spectrum /= np.sqrt(np.mean(shape_spectrum**2))

        noise_spectrum *= shape_spectrum

    return np.fft.irfft(noise_spectrum);

BLUE = lambda x: np.sqrt(x)
VIOLET = lambda x: x
BROWN = lambda x: 1/np.where(x==0, np.inf, x)
PINK = lambda x: 1/np.where(x==0, np.inf, np.sqrt(x))

Different noise colours have noticeably different sounds. None of them are particularly nice to listen to on their own, but pink and brown are a bit mellower than those with more high frequencies.

In [ ]:
play(noise(duration=2))
play(noise(duration=2, shape=PINK))
play(noise(duration=2, shape=BROWN))
play(noise(duration=2, shape=BLUE))
play(noise(duration=2, shape=VIOLET))

(Plotting the waveforms and spectra is left as an exercise for the reader. You saw on Monday what these things look like -- it's not super interesting.)

# Envelope

So far all the sounds we've made are pretty dull because they just stay the same over time. Real sounds evolve and change in all kinds of ways. Some of the these can be really complicated, but you can do quite a bit with just a simple **amplitude envelope**: varying the volume of the sound across its duration.

A simple but effective and very widely used envelope is **ADSR**, which divides the sound into four phases:

* **Attack**: the initial onset of the sound, as it goes from silence to full volume, defined as a duration. A short attack means the sound starts abruptly, whereas with a long attack it gradually fades in.
* **Decay**: a relaxation from the full volume of the attack to the level the sound will stay at if held. This is again expressed in terms of time.
* **Sustain**: a level at which the sound will settle after the decay and stay at for as long as the note is held. This is expressed as a fraction of the maximum volume (which is 1).
* **Release**: how long the note takes to fade back to silence once it is no longer being held. Again this is defined in terms of time.


In [ ]:
def envelope(duration=DURATION, rate=SAMPLE_RATE,
             attack=0.01, decay=0.05, sustain=0, release=0):
    """
    Simple linear ADSR envelope. Attack, decay and sustain are
    specified in seconds, generated according to the rate. Sustain
    fills everything not taken up by the other phases. If the
    duration is insufficient to contain all the elements the
    excess is discarded.
    """
    N = int(duration * rate)
    result = np.full(N, sustain, dtype=float)

    nR = np.min((int(release * rate), N))
    if nR:
        result[(N - nR):] = np.linspace(sustain, 0, nR)

    nA = int(attack * rate)
    if nA:
        result[:nA] = np.linspace(0, 1, nA)

    nD = np.min((int(decay * rate), N-nA))
    if nD:
        result[nA:(nA + nD)] = np.linspace(1, result[np.min((N, nA+nD))], nD)

    return result

Here's a plot of an envelope to illustrate the shape.

In [ ]:
env = envelope(0.1, attack=0.02, decay=0.02, sustain=0.5, release=0.03)
tt = np.linspace(0, 0.1, len(env))
plt.plot(tt, env)
plt.xlabel("Time")
plt.ylabel("Amplitude")
for xx in [0, 0.02, 0.04, 0.07, 0.1]:
  plt.axvline(x=xx, color='gray', linestyle=":")
plt.axhline(0, color='gray', linestyle=":");


# Sounds

Let's put all these ingredients together to synthesise some simple sounds. We'll use a mix of noise and sawtooth waves for something vaguely percussive, square waves for something slightly more tonal.

In [ ]:
# for simplicity, we'll make all the sounds the same length
# half a second is the duration of a quarter note at 120 bpm
QUARTER = 0.5

KICK = noise(duration=QUARTER, shape=PINK) * envelope(duration=QUARTER) + saw(55, duration=QUARTER) * envelope(decay=0.2, duration=QUARTER)
TINK = noise(duration=QUARTER, shape=BLUE) * envelope(duration=QUARTER) + saw(4400, duration=QUARTER) * envelope(decay=0.1, duration=QUARTER)
BEEP = square(330, duration=QUARTER) * envelope(attack=0.0, decay=0.2, duration=QUARTER)
BUZZ = square(44, duration=QUARTER) * envelope(attack=0.05, decay=0.2, duration=QUARTER)

play(KICK)
play(TINK)
play(BEEP)
play(BUZZ)

Okay, not the most convincing drums ever, but enough to make a little rhythm loop.

In [ ]:
BEAT = np.concatenate((KICK, TINK, TINK, TINK))

play(BEAT)

# Music*

<small>(* in an extremely loose sense)</small>

To go along with the beat, let's add some chords for harmony. Specifying these via frequency is bit laborious, so we'll add some utilities to map note names to frequency. (Don't worry about the mechanics of this for now. If you know nothing about notes and chords that's completely fine.)

In [ ]:
# basic map of note names to offsets within an octave
BASE_NOTES = { 'C' : 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11 }
FLATS = { f'{key}b' : (val - 1) for key,val in BASE_NOTES.items() }
SHARPS = { f'{key}#' : (val + 1) for key,val in BASE_NOTES.items() }
NAMES = { **BASE_NOTES, **FLATS, **SHARPS }

def note(name):
  """
  Convert a named note with octave (like 'C#3')
  to a note number.
  """
  octave = int(name[-1])
  offset = NAMES[name[:-1]]
  return (octave + 1) * 12 + offset

def tet_12 ( name, base='A4', base_freq=440 ):
  """
  Determine the frequency of a named note in 12-tone equal temperament tuning.
  """
  return base_freq * 2**((note(name) - note(base))/12)

def chord( names, base='A4', base_freq=440 ):
  """
  Convert a string containing a set of notes
  separated by whitespace into a list of frequencies
  suitable (inter alia) for passing to the tones function.
  """
  return [ tet_12(name, base, base_freq) for name in names.split() ]

Now we can use these functions to play some chords. We'll just use the `tones` function for this, producing an equal mix of simple sine
waves. The result should sound vaguely like an organ.

**NB**: This is literally just some vaguely-in-tune random nonsense I typed in off the cuff, so don't expect anything very interesting. Here and below, you are strongly encouraged to play around and generate your own content. If there's time we may even listen to some of it in the tutorial 😃

In [ ]:
BAR = QUARTER * 4

# a banal A minor chord progression
# if you know what you're doing -- or even if you don't -- feel free to change this up
# just remember that when we're applying envelopes and (later on) mixing stuff
# together, the signals need to be of consistent sizes
CHORD_NOTES = ["A2 C3 E3 A3", "C3 E3 G3 C3", "E3 G3 B3 D4", "D3 F3 A3 D4"]

# each chord has an envelope
PAD_ENV = envelope(duration=BAR, attack=0.2, decay=0.5, sustain=0.5, release=0.5)
CHORDS = [ PAD_ENV * tones(chord(cc), duration=BAR) for cc in CHORD_NOTES ]

# and we also add a longer envelope over the whole phrase
PAD = np.concatenate(CHORDS) * envelope(duration=BAR * len(CHORDS), attack=BAR, decay=BAR, sustain=0.75, release=BAR)

play(PAD)

Let's try mixing this chord sequence with the rhythm loop. Mixing is just taking a weighted sum of the two signals -- but they need to be the same length.

In [ ]:
# repeat the beat to be as long as the chord sequence
BEATS = np.concatenate([BEAT] * 4)

# take a weighted sum
mix = 0.5 * BEATS + 0.4 * PAD
play(mix)

Finally, let's add a simple melody line on top of this. This time we'll use a sawtooth wave as the basis for the note, and we'll again give it an envelope. We'll build the sequence out of eighth notes (half as long as the drum beats, one eighth the length of the chords) so we can mix up the rhythm a tiny bit.

Again, this would be tedious to do in terms of frequencies and durations, so we'll make some helper functions that allow is to just write a list of notes.

In [ ]:
EIGHTH = QUARTER / 2
MELODY_ENV = envelope(EIGHTH, attack=0.01, decay=0.05, sustain=0.5, release=0.1)

def melody_note(name, duration=EIGHTH, env=MELODY_ENV, wavefunc=saw, rate=SAMPLE_RATE):
  """
  Generate a single sawtooth note by name.
  In addition to the names defined earlier, we recognise
  `r` (case insensitive) to denote a rest, for which we'll
  return the appropriate duration of silence.
  """
  return np.zeros(int(duration * rate)) if name.lower()=='r' else env * wavefunc(tet_12(name), duration=duration)

def melody(names, duration=EIGHTH, env=MELODY_ENV, wavefunc=saw, rate=SAMPLE_RATE):
  """
  Take a whitespace-delimted string of note names and rests
  and return the generated audio signal.
  """
  return np.concatenate([melody_note(name, duration, env, wavefunc, rate) for name in names.split()])

And now for the list of notes.

(As with the chords, this is just some random stuff off the top of my head, it doesn't qualify as a tune. Feel free to play around with it however you like.)

In [ ]:
MELODY_NOTES = '''
A5 A5 r  A4 B4 C5  r  E5
C5 C4 E5 r  E5 G4  r  D#5
E5 r  G5 r  E5 D#5 D5 r
D5 r  D5 r  r  C#5 C5 r
'''

MELODY = melody(MELODY_NOTES)
play(MELODY)

The enveloped saw produces a sort of chiptune-y sound, but it's pretty thin on its own. Let's mix it with the beats and chords.

In [ ]:
mix = 0.4 * BEATS + 0.3 * PAD + 0.3 * MELODY
play(mix)

# Effects

Layering the different elements gives the result more depth and structure, but there's still not much to it. Bearing in mind the ideas of **subtractive synthesis**, we might consider shaping the sound with some **filters** and **effects**.

While it is possible to implement these explicitly using basic NumPy functions, doing so is a bit of a chore. Instead, let's make use of functionality already implemented in Pedalboard.

For example, we could use a **low pass filter** to smooth out the melody sound.

In [ ]:
ladder = pedalboard.LadderFilter(pedalboard.LadderFilter.Mode.LPF12, cutoff_hz=1000, resonance=0.6, drive=5)
filtered = ladder.process(MELODY, SAMPLE_RATE)
play(MELODY)
play(filtered)


That's removed a lot of the roughness, but the result seems a bit bland. Maybe we could go the other way and add some **distortion**.

(**NB**: these will be a fair bit louder than the previous, so be careful with your headphone volume.)

In [ ]:
overdrive = pedalboard.Distortion(drive_db=20)
driven = overdrive.process(MELODY, SAMPLE_RATE)
driven_filtered = overdrive.process(filtered, SAMPLE_RATE)
play(driven)
play(driven_filtered)

I quite like the combo, but I'd like to give it a bit more movement. Let's try a **chorus** effect. This is a modulated short delay that sort-of emulates having multiple performers each giving slightly different performances.

In [ ]:
chorus = pedalboard.Chorus(rate_hz=3, depth=0.1, feedback=0.75, mix=0.25)
lead = chorus.process(driven_filtered, SAMPLE_RATE)

play(lead)

Ok, that's hideous but let's go with it. What about the other tracks?

Let's try a **phaser** on the harmony. As mentioned in the lecture, this is an effect based on **all-pass filters**, which shift the phases of different frequencies without changing their magnitude.

In [ ]:
phaser = pedalboard.Phaser(rate_hz=0.33, depth=0.5, feedback=0.25, mix=0.5)
harmony = phaser.process(PAD, SAMPLE_RATE)

play(harmony)

As for the drums, let's throw in a bit of **delay**: adding back a bit of the past signal to get an echo effect.

In [ ]:
echo = pedalboard.Delay(delay_seconds=0.25, feedback=0.2, mix=0.2)
drums = echo.process(BEATS, SAMPLE_RATE)

play(drums)


All of these are pretty cheesy, but that's fine. Feel free to tweak the parameters or switch things around however you like.

In the meantime, let's put things back together and see what we've got.

In [ ]:
mix = 0.5 * drums + 0.25 * harmony + 0.1 * lead
play(mix)

To finish things off, let's throw on some **reverb**. Everything sounds better with reverb.

Reverb is a class of effects that emulate how sounds behave in spaces. There are several kinds of reverb, and we'll come back to this topic in a few weeks, but for now we'll just use a basic **parametric reverb** for a big room sound.

**NB:** Reverb has a "tail" -- the reverberations continue for some time after the sound finishes. (Delays do this too, but let's not worry about that.) Effect processing in Pedalboard returns something the same length you gave it, so we'll add a bit of silence at the end of the mix to leave space for the tail.

In [ ]:
reverb = pedalboard.Reverb(room_size=0.75, damping=0.5, wet_level=0.5, dry_level=0.5)
play(reverb.process(np.concatenate([mix, np.zeros(SAMPLE_RATE * 2)]), SAMPLE_RATE))

# Envoi

You will probably not do this kind of thing in this way often -- or at all -- in your future endeavours. And that's fine. This tutorial is absolutely not telling you how to do things. It is only intended to illustrate some of the underlying processes of digital audio synthesis, which can be a bit fiddly but are not really all that complicated.

For now, if you have the time and inclination, play around with the code and see what you come up with.